In [ ]:
# Data-path configuration (see ../paths.py and ../DATA.md).
# Set the SCREAM_AUTOTUNE_DATA env var to point at the unpacked Zenodo
# data deposit; defaults to <repo>/data. Assumes this notebook is run
# from its own directory (the Jupyter default).
import os, sys
sys.path.insert(0, os.path.abspath(".."))
import paths

# Preprocessing surrogate training workflow


Load in packages - Tensorflow warnings may affect ESEM GP performance

In [ ]:
import os
import json
import math
import pandas as pd
import numpy as np
import xarray as xr
import glob
import pickle
from esem import gp_model
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import KFold, cross_val_score, cross_validate
from sklearn.metrics import make_scorer, r2_score, root_mean_squared_error
from sklearn import preprocessing
from sklearn.pipeline import make_pipeline

import tensorflow as tf
import gpflow
from datetime import date
from datetime import datetime

from scipy.optimize import minimize

## Running the models

### Load the files/parameters/variables

Load parameter sampling

In [ ]:
path_to_json = str(paths.PPE_PARAMS_JSON) #this file contains the specific parameters for each run
ppe_params_all = pd.read_json(path_to_json)

In [ ]:
(ppe_params_all['p3_ice_sed_knob']>=1).sum() #number of ppe runs with required restricted parameter ranges

In [ ]:
##collects the labels for all runs --- DY2
DY2_path = str(paths.DY2_DIR) + "/" #path to the runs
DY2_missing_folders =[] #will collect all removed runs
DY2_folders = []

#Reading in all file folders
#m**** files
    #When trying increasing amounts of data we will go with up to 50, 100, 150, 200, 287
for m in range(0, 301): #DY1 goes to 301
    folder = 'm{:04}'.format(m)
    if os.path.exists(DY2_path+folder):
        DY2_folders.append(m)
    else:
        DY2_missing_folders.append(m) #just in integers, not file name (if you want this append folder)  
DY2_folders = ['m{:04}'.format(m) for m in DY2_folders] #writes the runs in the m**** format
#opt**** files
for file in os.listdir(DY2_path):
    #if file.startswith("m0") or file.startswith("opt"):
    if file.startswith("opt"):
        DY2_folders.append(file)
#does not include t0000 as this is pointed to by m0000

#removing files that don't have all the data--could be doing this better
DY2_folders.remove('m0230') #this file is missing 3hr averages
DY2_folders.remove('optmar20day5') #this file is missing 2nd day daily averages

#not in DY1
DY2_folders.remove('m0024')
DY2_folders.remove('m0025')
DY2_folders.remove('m0061')
DY2_folders.remove('optmar22hd')
DY2_folders.remove('optmar15')
DY2_folders.remove('optmar20day2')

print(len(DY2_folders)) 
DY2_folders = sorted(DY2_folders)

##adds the path name for all files--only 2nd day, daily averages
DY2_filename_list = []
DY2_data_dir = DY2_path #'/global/cfs/projectdirs/e3smdata/simulations/ecp-autotune/SCREAM.2024-autocal-00.ne1024pg2/'
#data_dir = '/global/cfs/projectdirs/e3smdata/simulations/SCREAM.2024-autocal-00.ne1024pg2/' #pointed to by above
for f in DY2_folders:
    file_path = DY2_data_dir+f+'/SCREAM.2024-autocal-00.ne1024pg2/run/output.scream.AutoCal.daily_avg_ne30pg2.AVERAGE.nhours_x24.2020-01-26-00000.nc'
    #file_path1 = data_dir+f+'/SCREAM.2024-autocal-00.ne1024pg2/run/output.scream.AutoCal.daily_avg_ne30pg2.AVERAGE.nhours_x24.2020-01-20-00000.nc'
    #file_path2 = data_dir+f+'/SCREAM.2024-autocal-00.ne1024pg2/run/output.scream.AutoCal.daily_avg_ne30pg2.AVERAGE.nhours_x24.2020-01-26-00000.nc'
    DY2_filename_list.append(file_path)

#count_fail = 0
##filter ice_sed < 1 to exclude runs outside of typical range
DY2_file_check = np.zeros(len(DY2_filename_list),dtype=bool) #initialize file check
for i in range(len(DY2_filename_list)):
    ppe_member = DY2_folders[i]
    if (float(ppe_params_all['p3_ice_sed_knob'][ppe_member]) >= 1.0):
        DY2_file_check[i] = True
#    else:
#        count_fail += 1
#print(count_fail)

DY2_filename_list_filtered = np.array(DY2_filename_list)[DY2_file_check==True] #turns the file name list to a np array
DY2_sim_names = np.array(DY2_folders)[DY2_file_check==True]
DY2_ppe_params = ppe_params_all[ppe_params_all.index.isin(DY2_sim_names)]
len(DY2_ppe_params)

In [ ]:
##collects the labels for all runs --- DY1
DY1_path = str(paths.DY1_DIR) + "/" #new scratch location of DY1
#"/global/cfs/cdirs/e3smdata/simulations/ecp-autotune/sims-s15-mar7/setupA1/" #previous path to the runs
DY1_missing_folders =[] #will collect all removed runs
DY1_folders = []

#When trying increasing amounts of data we will go with up to 50, 100, 150, 200, 287 (this is where DY2 ends)
for m in range(0, 301): 
    folder = 'm{:04}'.format(m)
    if os.path.exists(DY1_path+folder):
        DY1_folders.append(m)
    else:
        DY1_missing_folders.append(m) #just in integers, not file name (if you want this append folder)  
DY1_folders = ['m{:04}'.format(m) for m in DY1_folders] #writes the runs in the m**** format

#Could also pull all files in this way, but will be out of order and include m0000-5day
for file in os.listdir(DY1_path):
    #if file.startswith("m0") or file.startswith("opt"):
    if file.startswith("opt"):
        DY1_folders.append(file)
#does not include t0000 as this is pointed to by m0000

#removing files that don't have all the data
#folders.remove('m0230') #this file is missing 3hr averages
#folders.remove('optmar20day5') #this file is missing 2nd day daily averages

#DY2 -- invalid runs -- also not in the json
DY1_folders.remove('m0024')
DY1_folders.remove('m0025')
DY1_folders.remove('m0061')
DY1_folders.remove('optmar22hd')

DY1_folders.remove('m0262')
DY1_folders.remove('m0263')
DY1_folders.remove('m0264')
DY1_folders.remove('m0266')
DY1_folders.remove('m0267')
DY1_folders.remove('m0270')
DY1_folders.remove('m0272')
DY1_folders.remove('m0274')
DY1_folders.remove('m0275')
DY1_folders.remove('m0279')
DY1_folders.remove('m0289')
DY1_folders.remove('m0290')
DY1_folders.remove('m0292')
DY1_folders.remove('m0293')
DY1_folders.remove('m0294')
DY1_folders.remove('m0295')
DY1_folders.remove('m0296')
DY1_folders.remove('m0299')
DY1_folders.remove('m0300')

DY1_folders.remove('optmar15seed0')
DY1_folders.remove('optmar27a')
DY1_folders.remove('optmar20dayAll')
DY1_folders.remove('optmar20day2-fail')
DY1_folders.remove('optmar15b')
DY1_folders.remove('optmar20day2-ltend')

#Not in DY2
DY1_folders.remove('m0230')
DY1_folders.remove('optmar20day5')
print(len(DY1_folders))
DY1_folders = sorted(DY1_folders)

##adds the path name for all files--only 2nd day, daily averages
DY1_filename_list = []
DY1_data_dir = DY1_path #'/global/cfs/cdirs/e3smdata/simulations/ecp-autotune/sims-s15-mar7/setupA1/'
#data_dir = '/global/cfs/projectdirs/e3smdata/simulations/SCREAM.2024-autocal-00.ne1024pg2/' #pointed to by above
for f in DY1_folders:
    file_path = DY1_data_dir+f+'/SCREAM.2024-autocal-00.ne1024pg2/run/output.scream.AutoCal.daily_avg_ne30pg2.AVERAGE.nhours_x24.2016-08-07-00000.nc'
    DY1_filename_list.append(file_path)
    
##filter ice_sed < 1 to exclude runs outside of typical range
DY1_file_check = np.zeros(len(DY1_filename_list),dtype=bool) #initialize file check
for i in range(len(DY1_filename_list)):
    ppe_member = DY1_folders[i]
    #if (int(sim_names[i][1:]) < 265): #try it with all the runs
    if (float(ppe_params_all['p3_ice_sed_knob'][ppe_member]) >= 1.0):
        DY1_file_check[i] = True

DY1_filename_list_filtered = np.array(DY1_filename_list)[DY1_file_check==True] #turns the file name list to a np array
DY1_sim_names = np.array(DY1_folders)[DY1_file_check==True]
DY1_ppe_params = ppe_params_all[ppe_params_all.index.isin(DY1_sim_names)]
len(DY1_ppe_params)

In [ ]:
#Check for intersection between DY1 and DY2, take only runs in both
print(list(set(DY1_sim_names) - set(DY2_sim_names))) # should be empty
print(list(set(DY2_sim_names) - set(DY1_sim_names)))
sim_names = [sim for sim in DY1_sim_names if sim in DY2_sim_names] 

ppe_params = ppe_params_all.loc[sim_names]
ppe_params

In [ ]:
# Open and concatenate the datasets along the new 'run_number' dimension
DY1_concat_dataset = xr.open_mfdataset(DY1_filename_list_filtered, concat_dim='run_label', combine='nested')
DY1_sim_names_from_paths = [path.split('/')[-4] for path in DY1_filename_list_filtered]
DY1_ppe_dataset_time = DY1_concat_dataset.assign_coords(run_label=('run_label', DY1_sim_names_from_paths))
#DY1_ppe_dataset_time = DY1_concat_dataset.assign_coords(run_label=('run_label', DY1_sim_names)) # Assign the 'run_number' coordinate
DY1_ppe_dataset = DY1_ppe_dataset_time.squeeze('time')


In [ ]:
# Open and concatenate the datasets along the new 'run_number' dimension
DY2_concat_dataset = xr.open_mfdataset(DY2_filename_list_filtered, concat_dim='run_label', combine='nested')
DY2_sim_names_from_paths = [path.split('/')[-4] for path in DY2_filename_list_filtered]
DY2_ppe_dataset_time = DY2_concat_dataset.assign_coords(run_label=('run_label', DY2_sim_names_from_paths))
#DY2_ppe_dataset_time = DY2_concat_dataset.assign_coords(run_label=('run_label', DY2_sim_names)) # Assign the 'run_number' coordinate
DY2_ppe_dataset = DY2_ppe_dataset_time.squeeze('time')


##### Observations

In [ ]:
DY1_obs_dir = str(paths.DY1_OBS_DIR) + '/'
DY1_precip_obs_file = DY1_obs_dir + 'IMERG.precip_total_surf_mass_flux.daily_AVERAGE.ne30pg2.20160807_mahf708.nc' #maybe check this
DY1_LW_obs_file = DY1_obs_dir + 'CERES.LW_flux_up_at_model_top.daily_AVERAGE.ne30pg2.20160807_mahf708.nc'
#DY1_SW_dn_obs_file = DY1_obs_dir + 'CERES.SW_flux_dn_at_model_top.daily_AVERAGE.ne30pg2.20160807_mahf708.nc'
DY1_SW_up_obs_file = DY1_obs_dir + 'CERES.SW_flux_up_at_model_top.daily_AVERAGE.ne30pg2.20160807_mahf708.nc'
DY1_LWP_file = DY1_obs_dir + 'mac.clwp-tlwp-wvp.20160807.ne30pg2.nc'

DY1_PCP_obs = xr.open_dataset(DY1_precip_obs_file).variables['precip_total_surf_mass_flux'].squeeze('time')
DY1_TLWP_obs = (xr.open_dataset(DY1_LWP_file).variables['tlwp']*1e-3).squeeze('time')
DY1_OSR_obs = xr.open_dataset(DY1_SW_up_obs_file).variables['SW_flux_up_at_model_top'].squeeze('time')
DY1_OLR_obs = xr.open_dataset(DY1_LW_obs_file).variables['LW_flux_up_at_model_top'].squeeze('time')

In [ ]:
DY2_obs_dir = str(paths.DY2_OBS_DIR) + '/'
DY2_precip_obs_file = DY2_obs_dir + 'IMERG.precip_total_surf_mass_flux.AVERAGE.ne30pg2.20200126.nc'
DY2_LW_obs_file = DY2_obs_dir + 'CERES.LW_flux_up_at_model_top.AVERAGE.ne30pg2.20200126.nc'
#DY2_SW_dn_obs_file = DY2_obs_dir + 'CERES.SW_flux_dn_at_model_top.AVERAGE.ne30pg2.20200126.nc'
DY2_SW_up_obs_file = DY2_obs_dir + 'CERES.SW_flux_up_at_model_top.AVERAGE.ne30pg2.20200126.nc'
DY2_LWP_file = DY2_obs_dir + 'mac.clwp-tlwp-wvp.20200126.ne30pg2.nc'

DY2_PCP_obs = xr.open_dataset(DY2_precip_obs_file).variables['precip_total_surf_mass_flux'].squeeze('time')
DY2_TLWP_obs = (xr.open_dataset(DY2_LWP_file).variables['tlwp']*1e-3).squeeze('time')
DY2_OSR_obs = xr.open_dataset(DY2_SW_up_obs_file).variables['SW_flux_up_at_model_top'].squeeze('time')
DY2_OLR_obs = xr.open_dataset(DY2_LW_obs_file).variables['LW_flux_up_at_model_top'].squeeze('time')

##### Filter data

In [ ]:
#filter variables in output - keep only the 4 of interest
to_keep = ['precip_total_surf_mass_flux','LiqWaterPath','RainWaterPath','SW_flux_up_at_model_top','LW_flux_up_at_model_top']
           
to_leave = ['SW_flux_dn','SW_flux_dn_at_model_bot','SW_flux_up','SW_flux_up_at_model_bot','SW_flux_dn_at_model_top', 'T_2m',
'T_mid', 'precip_ice_surf_mass_flux','precip_liq_surf_mass_flux','ps','qc','qi','qm','qr','qv','qv_2m','LW_flux_up','LW_flux_up_at_model_bot',
'IceWaterPath', 'LW_flux_dn','LW_flux_dn_at_model_bot','time_bnds', 'LongwaveCloudForcing', 'MeridionalVapFlux','ShortwaveCloudForcing','U','V', 
'VapWaterPath', 'ZonalVapFlux','bm','eddy_diff_mom', 'eff_radius_qc_at_cldtop','eff_radius_qi_at_cldtop', 'homme_T_mid_tend', 'homme_qv_tend',
'horiz_winds_at_model_bot', 'nc', 'ni', 'nr', 'omega', 'p3_T_mid_tend', 'p3_qv_tend', 'rrtmgp_T_mid_tend', 'sgs_buoy_flux', 'shoc_T_mid_tend',
'shoc_qv_tend', 'surf_evap', 'surf_mom_flux', 'surf_radiative_T','surf_sens_flux', 'surface_upward_latent_heat_flux', 'avg_count_ncol', 
'avg_count_ncol_lev', 'avg_count_ncol_dim', 'area', 'lat', 'lon']

In [ ]:
DY1_ppe_dataset_small = DY1_ppe_dataset.drop_vars(to_leave)
DY1_ppe_dataset_small['TotalLiqWaterPath'] = (DY1_ppe_dataset_small.LiqWaterPath + DY1_ppe_dataset_small.RainWaterPath)
#ppe_dataset_small['precip_total_surf_mass_flux'] = ppe_dataset_small['precip_total_surf_mass_flux']*1e-3*24*3600
DY1_ppe_dataset_small = DY1_ppe_dataset_small.drop_vars('p_levs')
DY1_ppe_dataset_small = DY1_ppe_dataset_small.rename({var: f"DY1_{var}" for var in DY1_ppe_dataset_small.data_vars})

In [ ]:
DY2_ppe_dataset_small = DY2_ppe_dataset.drop_vars(to_leave)
DY2_ppe_dataset_small['TotalLiqWaterPath'] = (DY2_ppe_dataset_small.LiqWaterPath + DY2_ppe_dataset_small.RainWaterPath)
#ppe_dataset_small['precip_total_surf_mass_flux'] = ppe_dataset_small['precip_total_surf_mass_flux']*1e-3*24*3600]
#DY2_ppe_dataset_small = DY2_ppe_dataset_small.drop_vars('p_levs')
DY2_ppe_dataset_small = DY2_ppe_dataset_small.rename({var: f"DY2_{var}" for var in DY2_ppe_dataset_small.data_vars})

In [ ]:
assert list(ppe_params.index) == sim_names
assert list(DY1_ppe_dataset_small.run_label.values) == sim_names
assert list(DY2_ppe_dataset_small.run_label.values) == sim_names

#### Taking spatial averages - zonal, regional, global

##### Mask to observations
Observations are available on a subset of grid cells; this will affect averages, so first mask to the subset of grid cells

In [ ]:
##two ways to get these masks
mask_path =str(paths.MASK_DIR) + '/'

DY1_PCP_mask = xr.open_dataset(mask_path+'/precip_1_output.nc').squeeze('time', drop=True)
DY2_PCP_mask = xr.open_dataset(mask_path+'/precip_2_output.nc').squeeze('time', drop=True)
DY1_TLWP_mask = xr.open_dataset(mask_path+'/tlwp_1_output.nc').squeeze('time', drop=True)
DY2_TLWP_mask = xr.open_dataset(mask_path+'/tlwp_2_output.nc').squeeze('time', drop=True)

#len(np.where(((DY1_TLWP_mask.to_dataarray()[0]) == True).values)[0]) - ncol drops from 21600 to 15313

In [ ]:
DY1_ppe_dataset_mask = DY1_ppe_dataset_small.copy(deep=True)
DY2_ppe_dataset_mask = DY2_ppe_dataset_small.copy(deep=True)

#DY1
DY1_ppe_dataset_mask['DY1_precip_total_surf_mass_flux'] = DY1_ppe_dataset_small.DY1_precip_total_surf_mass_flux.where(np.isnan(DY1_PCP_obs) == False)
DY1_ppe_dataset_mask['DY1_TotalLiqWaterPath'] = DY1_ppe_dataset_small.DY1_TotalLiqWaterPath.where(np.isnan(DY1_TLWP_obs) == False)
DY1_ppe_dataset_mask['DY1_SW_flux_up_at_model_top'] = DY1_ppe_dataset_small.DY1_SW_flux_up_at_model_top.where(np.isnan(DY1_OSR_obs) == False)
DY1_ppe_dataset_mask['DY1_LW_flux_up_at_model_top'] = DY1_ppe_dataset_small.DY1_LW_flux_up_at_model_top.where(np.isnan(DY1_OLR_obs) == False)

#DY2
DY2_ppe_dataset_mask['DY2_precip_total_surf_mass_flux'] = DY2_ppe_dataset_small.DY2_precip_total_surf_mass_flux.where(np.isnan(DY2_PCP_obs) == False)
DY2_ppe_dataset_mask['DY2_TotalLiqWaterPath'] = DY2_ppe_dataset_small.DY2_TotalLiqWaterPath.where(np.isnan(DY2_TLWP_obs) == False)
DY2_ppe_dataset_mask['DY2_SW_flux_up_at_model_top'] = DY2_ppe_dataset_small.DY2_SW_flux_up_at_model_top.where(np.isnan(DY2_OSR_obs) == False)
DY2_ppe_dataset_mask['DY2_LW_flux_up_at_model_top'] = DY2_ppe_dataset_small.DY2_LW_flux_up_at_model_top.where(np.isnan(DY2_OLR_obs) == False)

In [ ]:
regions_file = xr.open_dataset(str(paths.REGIONS_FILE))
regions_list = ['poles','extratropical_land','extratropical_ocean','tropical_land','ascending_tropical_ocean','descending_tropical_ocean']
#area = ppe_dataset.area[1,:] #only taking the first row, because all rows should have the same values
control = xr.open_dataset(str(paths.CONTROL_FILE))
area = control.variables['area'][:]
lat = control.variables['lat'][:]
lon = control.variables['lon'][:]

In [ ]:
def zonal_means_native(data, area, lat, lon):
    data = data.squeeze()
    masked_area = np.where(np.isnan(data), np.nan, area).squeeze()
    lat = lat.squeeze()
    lat_bands = np.linspace(-90, 90, 19) #currently dividing globe in 18 zones via 19 borders - 10 degree bands
    zonal_means = dict()
    for i in range(len(lat_bands) - 1):
        mask_zone = (lat >= lat_bands[i]) & (lat < lat_bands[i+1]) #includes lower bound in the band (! might cause missing north pole)
        data_zone = np.where(mask_zone, data, np.nan)
        area_zone = np.where(mask_zone, masked_area, np.nan)
        zone_center = lat_bands[i] + (lat_bands[i+1] - lat_bands[i])/2
        if np.all(np.isnan(data_zone)) or np.nansum(area_zone) == 0:
            zonal_means[zone_center] = np.nan
        else:
            zone_mean = np.nansum(data_zone * area_zone) / np.nansum(area_zone)
            zonal_means[zone_center] = zone_mean
    return zonal_means

def regional_means_native(data, area, region_data):  
    #region_data = xr.open_dataset('/global/cfs/projectdirs/e3smdata/simulations/ecp-autotune/regions.nc') #for performance improvement move region_data out as argument
    data = data.squeeze()
    masked_area = np.where(np.isnan(data), np.nan, area).squeeze()
    regions_list = ['poles', 'extratropical_land', 'extratropical_ocean', 
                    'tropical_land', 'ascending_tropical_ocean', 'descending_tropical_ocean']
    region_means = dict()
    for reg_name in regions_list:
        mask_reg = region_data[reg_name].squeeze()
        data_reg = np.where(mask_reg > 0, data, np.nan)
        area_reg = np.where(mask_reg > 0, masked_area, np.nan)
        if np.all(np.isnan(data_reg)) or np.nansum(area_reg) == 0:
            region_means[reg_name] = np.nan
        else:
            reg_mean = np.nansum(data_reg * area_reg) / np.nansum(area_reg)
            region_means[reg_name] = reg_mean
    return region_means

def global_means_native(data, area): #takes global averages
    data = data.squeeze()
    masked_area = np.where(np.isnan(data), np.nan, area).squeeze()
    global_mean = np.nansum(data*masked_area)/np.nansum(masked_area)
    return global_mean

In [ ]:
plt.scatter(lon, lat, c=DY1_TLWP_obs)

In [ ]:
plt.scatter(lon, lat, c=DY1_PCP_obs)

In [ ]:
DY1_PCP_zonal_data = dict()
DY1_TLWP_zonal_data = dict()
DY1_OSR_zonal_data = dict()
DY1_OLR_zonal_data = dict()

DY1_PCP_regional_data = dict()
DY1_TLWP_regional_data = dict()
DY1_OSR_regional_data = dict()
DY1_OLR_regional_data = dict()

DY1_PCP_global_data = []
DY1_TLWP_global_data = []
DY1_OSR_global_data = []
DY1_OLR_global_data = []

for run in sim_names:
    precip = DY1_ppe_dataset_mask.sel(run_label = run)['DY1_precip_total_surf_mass_flux']
    tlwp = DY1_ppe_dataset_mask.sel(run_label = run)['DY1_TotalLiqWaterPath']
    swflux = DY1_ppe_dataset_mask.sel(run_label = run)['DY1_SW_flux_up_at_model_top']
    lwflux = DY1_ppe_dataset_mask.sel(run_label = run)['DY1_LW_flux_up_at_model_top']

    DY1_PCP_zonal_data[run] = zonal_means_native(precip, area, lat, lon)
    DY1_TLWP_zonal_data[run] = zonal_means_native(tlwp, area, lat, lon)
    DY1_OSR_zonal_data[run] = zonal_means_native(swflux, area, lat, lon)
    DY1_OLR_zonal_data[run] = zonal_means_native(lwflux, area, lat, lon)
    
    DY1_PCP_regional_data[run] = regional_means_native(precip, area, regions_file)
    DY1_TLWP_regional_data[run] = regional_means_native(tlwp, area, regions_file)
    DY1_OSR_regional_data[run] = regional_means_native(swflux, area, regions_file)
    DY1_OLR_regional_data[run] = regional_means_native(lwflux, area, regions_file)    
    
    DY1_PCP_global_data.append(global_means_native(precip, area))
    DY1_TLWP_global_data.append(global_means_native(tlwp, area))
    DY1_OSR_global_data.append(global_means_native(swflux, area))
    DY1_OLR_global_data.append(global_means_native(lwflux, area))

In [ ]:
DY2_PCP_zonal_data = dict()
DY2_TLWP_zonal_data = dict()
DY2_OSR_zonal_data = dict()
DY2_OLR_zonal_data = dict()

DY2_PCP_regional_data = dict()
DY2_TLWP_regional_data = dict()
DY2_OSR_regional_data = dict()
DY2_OLR_regional_data = dict()

DY2_PCP_global_data = []
DY2_TLWP_global_data = []
DY2_OSR_global_data = []
DY2_OLR_global_data = []

for run in sim_names:
    precip = DY2_ppe_dataset_mask.sel(run_label = run)['DY2_precip_total_surf_mass_flux']
    tlwp = DY2_ppe_dataset_mask.sel(run_label = run)['DY2_TotalLiqWaterPath']
    swflux = DY2_ppe_dataset_mask.sel(run_label = run)['DY2_SW_flux_up_at_model_top']
    lwflux = DY2_ppe_dataset_mask.sel(run_label = run)['DY2_LW_flux_up_at_model_top']

    DY2_PCP_zonal_data[run] = zonal_means_native(precip, area, lat, lon)
    DY2_TLWP_zonal_data[run] = zonal_means_native(tlwp, area, lat, lon)
    DY2_OSR_zonal_data[run] = zonal_means_native(swflux, area, lat, lon)
    DY2_OLR_zonal_data[run] = zonal_means_native(lwflux, area, lat, lon)
    
    DY2_PCP_regional_data[run] = regional_means_native(precip, area, regions_file)
    DY2_TLWP_regional_data[run] = regional_means_native(tlwp, area, regions_file)
    DY2_OSR_regional_data[run] = regional_means_native(swflux, area, regions_file)
    DY2_OLR_regional_data[run] = regional_means_native(lwflux, area, regions_file)    
    
    DY2_PCP_global_data.append(global_means_native(precip, area))
    DY2_TLWP_global_data.append(global_means_native(tlwp, area))
    DY2_OSR_global_data.append(global_means_native(swflux, area))
    DY2_OLR_global_data.append(global_means_native(lwflux, area))

In [ ]:
#removing 3 zones because of NaNs - poles and -80-70 (-75)

In [ ]:
DY1_PCP_z_df = pd.DataFrame.from_dict(DY1_PCP_zonal_data, orient="index")
DY1_PCP_r_df = pd.DataFrame.from_dict(DY1_PCP_regional_data, orient="index")
DY1_PCP_zrg_ppedataset = pd.concat([DY1_PCP_z_df, DY1_PCP_r_df], axis=1)
DY1_PCP_zrg_ppedataset["global"] = DY1_PCP_global_data
DY1_PCP_zrg_ppedataset.columns = [f"DY1_{c}" for c in DY1_PCP_zrg_ppedataset.columns]

DY2_PCP_z_df = pd.DataFrame.from_dict(DY2_PCP_zonal_data, orient="index")
DY2_PCP_r_df = pd.DataFrame.from_dict(DY2_PCP_regional_data, orient="index")
DY2_PCP_zrg_ppedataset = pd.concat([DY2_PCP_z_df, DY2_PCP_r_df], axis=1)
DY2_PCP_zrg_ppedataset["global"] = DY2_PCP_global_data
DY2_PCP_zrg_ppedataset.columns = [f"DY2_{c}" for c in DY2_PCP_zrg_ppedataset.columns]

PCP_zrg_ppedataset = pd.concat([DY1_PCP_zrg_ppedataset, DY2_PCP_zrg_ppedataset], axis=1)

In [ ]:
DY1_TLWP_z_df = pd.DataFrame.from_dict(DY1_TLWP_zonal_data, orient="index")
DY1_TLWP_r_df = pd.DataFrame.from_dict(DY1_TLWP_regional_data, orient="index")
DY1_TLWP_zrg_ppedataset = pd.concat([DY1_TLWP_z_df, DY1_TLWP_r_df], axis=1)
DY1_TLWP_zrg_ppedataset["global"] = DY1_TLWP_global_data
DY1_TLWP_zrg_ppedataset.columns = [f"DY1_{c}" for c in DY1_TLWP_zrg_ppedataset.columns]

DY2_TLWP_z_df = pd.DataFrame.from_dict(DY2_TLWP_zonal_data, orient="index")
DY2_TLWP_r_df = pd.DataFrame.from_dict(DY2_TLWP_regional_data, orient="index")
DY2_TLWP_zrg_ppedataset = pd.concat([DY2_TLWP_z_df, DY2_TLWP_r_df], axis=1)
DY2_TLWP_zrg_ppedataset["global"] = DY2_TLWP_global_data
DY2_TLWP_zrg_ppedataset.columns = [f"DY2_{c}" for c in DY2_TLWP_zrg_ppedataset.columns]

TLWP_zrg_ppedataset = pd.concat([DY1_TLWP_zrg_ppedataset, DY2_TLWP_zrg_ppedataset], axis=1)

In [ ]:
DY1_OSR_z_df = pd.DataFrame.from_dict(DY1_OSR_zonal_data, orient="index")
DY1_OSR_r_df = pd.DataFrame.from_dict(DY1_OSR_regional_data, orient="index")
DY1_OSR_zrg_ppedataset = pd.concat([DY1_OSR_z_df, DY1_OSR_r_df], axis=1)
DY1_OSR_zrg_ppedataset["global"] = DY1_OSR_global_data
DY1_OSR_zrg_ppedataset.columns = [f"DY1_{c}" for c in DY1_OSR_zrg_ppedataset.columns]

DY2_OSR_z_df = pd.DataFrame.from_dict(DY2_OSR_zonal_data, orient="index")
DY2_OSR_r_df = pd.DataFrame.from_dict(DY2_OSR_regional_data, orient="index")
DY2_OSR_zrg_ppedataset = pd.concat([DY2_OSR_z_df, DY2_OSR_r_df], axis=1)
DY2_OSR_zrg_ppedataset["global"] = DY2_OSR_global_data
DY2_OSR_zrg_ppedataset.columns = [f"DY2_{c}" for c in DY2_OSR_zrg_ppedataset.columns]

OSR_zrg_ppedataset = pd.concat([DY1_OSR_zrg_ppedataset, DY2_OSR_zrg_ppedataset], axis=1)

In [ ]:
DY1_OLR_z_df = pd.DataFrame.from_dict(DY1_OLR_zonal_data, orient="index")
DY1_OLR_r_df = pd.DataFrame.from_dict(DY1_OLR_regional_data, orient="index")
DY1_OLR_zrg_ppedataset = pd.concat([DY1_OLR_z_df, DY1_OLR_r_df], axis=1)
DY1_OLR_zrg_ppedataset["global"] = DY1_OLR_global_data
DY1_OLR_zrg_ppedataset.columns = [f"DY1_{c}" for c in DY1_OLR_zrg_ppedataset.columns]

DY2_OLR_z_df = pd.DataFrame.from_dict(DY2_OLR_zonal_data, orient="index")
DY2_OLR_r_df = pd.DataFrame.from_dict(DY2_OLR_regional_data, orient="index")
DY2_OLR_zrg_ppedataset = pd.concat([DY2_OLR_z_df, DY2_OLR_r_df], axis=1)
DY2_OLR_zrg_ppedataset["global"] = DY2_OLR_global_data
DY2_OLR_zrg_ppedataset.columns = [f"DY2_{c}" for c in DY2_OLR_zrg_ppedataset.columns]

OLR_zrg_ppedataset = pd.concat([DY1_OLR_zrg_ppedataset, DY2_OLR_zrg_ppedataset], axis=1)

In [ ]:
PCP_zrg_ppedataset = PCP_zrg_ppedataset.drop(columns=['DY1_85.0', 'DY1_-85.0', 'DY1_-75.0', 'DY2_85.0', 'DY2_-85.0', 'DY2_-75.0'])
TLWP_zrg_ppedataset = TLWP_zrg_ppedataset.drop(columns=['DY1_85.0', 'DY1_-85.0', 'DY1_-75.0', 'DY2_85.0', 'DY2_-85.0', 'DY2_-75.0'])
OSR_zrg_ppedataset = OSR_zrg_ppedataset.drop(columns=['DY1_85.0', 'DY1_-85.0', 'DY1_-75.0', 'DY2_85.0', 'DY2_-85.0', 'DY2_-75.0'])
OLR_zrg_ppedataset = OLR_zrg_ppedataset.drop(columns=['DY1_85.0', 'DY1_-85.0', 'DY1_-75.0', 'DY2_85.0', 'DY2_-85.0', 'DY2_-75.0'])

In [ ]:
#### Ordering matters here
zrg_ppedataset = pd.concat([PCP_zrg_ppedataset, TLWP_zrg_ppedataset, OSR_zrg_ppedataset, OLR_zrg_ppedataset], axis=1) 

In [ ]:
assert list(ppe_params.index) == sim_names
assert list(zrg_ppedataset.index) == sim_names
assert list(DY1_ppe_dataset_small.run_label.values) == sim_names
assert list(DY2_ppe_dataset_small.run_label.values) == sim_names

In [ ]:
#calculating the geographic averages for observations as well
DY1_PCP_zonal_obs = dict()
DY1_TLWP_zonal_obs = dict()
DY1_OSR_zonal_obs = dict()
DY1_OLR_zonal_obs = dict()

DY1_PCP_regional_obs = dict()
DY1_TLWP_regional_obs = dict()
DY1_OSR_regional_obs = dict()
DY1_OLR_regional_obs = dict()

DY1_PCP_global_obs = []
DY1_TLWP_global_obs = []
DY1_OSR_global_obs = []
DY1_OLR_global_obs = []
   
DY1_PCP_zonal_obs['obs'] = zonal_means_native(DY1_PCP_obs, area, lat, lon)
DY1_TLWP_zonal_obs['obs'] = zonal_means_native(DY1_TLWP_obs, area, lat, lon)
DY1_OSR_zonal_obs['obs'] = zonal_means_native(DY1_OSR_obs, area, lat, lon)
DY1_OLR_zonal_obs['obs'] = zonal_means_native(DY1_OLR_obs, area, lat, lon)
    
DY1_PCP_regional_obs['obs'] = regional_means_native(DY1_PCP_obs, area, regions_file)
DY1_TLWP_regional_obs['obs'] = regional_means_native(DY1_TLWP_obs, area, regions_file)
DY1_OSR_regional_obs['obs'] = regional_means_native(DY1_OSR_obs, area, regions_file)
DY1_OLR_regional_obs['obs'] = regional_means_native(DY1_OLR_obs, area, regions_file)    
    
DY1_PCP_global_obs.append(global_means_native(DY1_PCP_obs, area))
DY1_TLWP_global_obs.append(global_means_native(DY1_TLWP_obs, area))
DY1_OSR_global_obs.append(global_means_native(DY1_OSR_obs, area))
DY1_OLR_global_obs.append(global_means_native(DY1_OLR_obs, area))

In [ ]:
DY2_PCP_zonal_obs = dict()
DY2_TLWP_zonal_obs = dict()
DY2_OSR_zonal_obs = dict()
DY2_OLR_zonal_obs = dict()

DY2_PCP_regional_obs = dict()
DY2_TLWP_regional_obs = dict()
DY2_OSR_regional_obs = dict()
DY2_OLR_regional_obs = dict()

DY2_PCP_global_obs = []
DY2_TLWP_global_obs = []
DY2_OSR_global_obs = []
DY2_OLR_global_obs = []
   
DY2_PCP_zonal_obs['obs'] = zonal_means_native(DY2_PCP_obs, area, lat, lon)
DY2_TLWP_zonal_obs['obs'] = zonal_means_native(DY2_TLWP_obs, area, lat, lon)
DY2_OSR_zonal_obs['obs'] = zonal_means_native(DY2_OSR_obs, area, lat, lon)
DY2_OLR_zonal_obs['obs'] = zonal_means_native(DY2_OLR_obs, area, lat, lon)
    
DY2_PCP_regional_obs['obs'] = regional_means_native(DY2_PCP_obs, area, regions_file)
DY2_TLWP_regional_obs['obs'] = regional_means_native(DY2_TLWP_obs, area, regions_file)
DY2_OSR_regional_obs['obs'] = regional_means_native(DY2_OSR_obs, area, regions_file)
DY2_OLR_regional_obs['obs'] = regional_means_native(DY2_OLR_obs, area, regions_file)    
    
DY2_PCP_global_obs.append(global_means_native(DY2_PCP_obs, area))
DY2_TLWP_global_obs.append(global_means_native(DY2_TLWP_obs, area))
DY2_OSR_global_obs.append(global_means_native(DY2_OSR_obs, area))
DY2_OLR_global_obs.append(global_means_native(DY2_OLR_obs, area))

In [ ]:
DY1_PCP_obs_z_df = pd.DataFrame.from_dict(DY1_PCP_zonal_obs, orient="index")
DY1_PCP_obs_r_df = pd.DataFrame.from_dict(DY1_PCP_regional_obs, orient="index")
DY1_PCP_zrg_obs = pd.concat([DY1_PCP_obs_z_df, DY1_PCP_obs_r_df], axis=1)
DY1_PCP_zrg_obs["global"] = DY1_PCP_global_obs
DY1_PCP_zrg_obs.columns = [f"DY1_{c}" for c in DY1_PCP_zrg_obs.columns]
DY2_PCP_obs_z_df = pd.DataFrame.from_dict(DY2_PCP_zonal_obs, orient="index")
DY2_PCP_obs_r_df = pd.DataFrame.from_dict(DY2_PCP_regional_obs, orient="index")
DY2_PCP_zrg_obs = pd.concat([DY2_PCP_obs_z_df, DY2_PCP_obs_r_df], axis=1)
DY2_PCP_zrg_obs["global"] = DY2_PCP_global_obs
DY2_PCP_zrg_obs.columns = [f"DY2_{c}" for c in DY2_PCP_zrg_obs.columns]
PCP_zrg_obs = pd.concat([DY1_PCP_zrg_obs, DY2_PCP_zrg_obs], axis=1)

DY1_TLWP_obs_z_df = pd.DataFrame.from_dict(DY1_TLWP_zonal_obs, orient="index")
DY1_TLWP_obs_r_df = pd.DataFrame.from_dict(DY1_TLWP_regional_obs, orient="index")
DY1_TLWP_zrg_obs = pd.concat([DY1_TLWP_obs_z_df, DY1_TLWP_obs_r_df], axis=1)
DY1_TLWP_zrg_obs["global"] = DY1_TLWP_global_obs
DY1_TLWP_zrg_obs.columns = [f"DY1_{c}" for c in DY1_TLWP_zrg_obs.columns]
DY2_TLWP_obs_z_df = pd.DataFrame.from_dict(DY2_TLWP_zonal_obs, orient="index")
DY2_TLWP_obs_r_df = pd.DataFrame.from_dict(DY2_TLWP_regional_obs, orient="index")
DY2_TLWP_zrg_obs = pd.concat([DY2_TLWP_obs_z_df, DY2_TLWP_obs_r_df], axis=1)
DY2_TLWP_zrg_obs["global"] = DY2_TLWP_global_obs
DY2_TLWP_zrg_obs.columns = [f"DY2_{c}" for c in DY2_TLWP_zrg_obs.columns]
TLWP_zrg_obs = pd.concat([DY1_TLWP_zrg_obs, DY2_TLWP_zrg_obs], axis=1)

DY1_OSR_obs_z_df = pd.DataFrame.from_dict(DY1_OSR_zonal_obs, orient="index")
DY1_OSR_obs_r_df = pd.DataFrame.from_dict(DY1_OSR_regional_obs, orient="index")
DY1_OSR_zrg_obs = pd.concat([DY1_OSR_obs_z_df, DY1_OSR_obs_r_df], axis=1)
DY1_OSR_zrg_obs["global"] = DY1_OSR_global_obs
DY1_OSR_zrg_obs.columns = [f"DY1_{c}" for c in DY1_OSR_zrg_obs.columns]
DY2_OSR_obs_z_df = pd.DataFrame.from_dict(DY2_OSR_zonal_obs, orient="index")
DY2_OSR_obs_r_df = pd.DataFrame.from_dict(DY2_OSR_regional_obs, orient="index")
DY2_OSR_zrg_obs = pd.concat([DY2_OSR_obs_z_df, DY2_OSR_obs_r_df], axis=1)
DY2_OSR_zrg_obs["global"] = DY2_OSR_global_obs
DY2_OSR_zrg_obs.columns = [f"DY2_{c}" for c in DY2_OSR_zrg_obs.columns]
OSR_zrg_obs = pd.concat([DY1_OSR_zrg_obs, DY2_OSR_zrg_obs], axis=1)

DY1_OLR_obs_z_df = pd.DataFrame.from_dict(DY1_OLR_zonal_obs, orient="index")
DY1_OLR_obs_r_df = pd.DataFrame.from_dict(DY1_OLR_regional_obs, orient="index")
DY1_OLR_zrg_obs = pd.concat([DY1_OLR_obs_z_df, DY1_OLR_obs_r_df], axis=1)
DY1_OLR_zrg_obs["global"] = DY1_OLR_global_obs
DY1_OLR_zrg_obs.columns = [f"DY1_{c}" for c in DY1_OLR_zrg_obs.columns]
DY2_OLR_obs_z_df = pd.DataFrame.from_dict(DY2_OLR_zonal_obs, orient="index")
DY2_OLR_obs_r_df = pd.DataFrame.from_dict(DY2_OLR_regional_obs, orient="index")
DY2_OLR_zrg_obs = pd.concat([DY2_OLR_obs_z_df, DY2_OLR_obs_r_df], axis=1)
DY2_OLR_zrg_obs["global"] = DY2_OLR_global_obs
DY2_OLR_zrg_obs.columns = [f"DY2_{c}" for c in DY2_OLR_zrg_obs.columns]
OLR_zrg_obs = pd.concat([DY1_OLR_zrg_obs, DY2_OLR_zrg_obs], axis=1)

PCP_zrg_obs = PCP_zrg_obs.drop(columns=['DY1_85.0', 'DY1_-85.0', 'DY1_-75.0', 'DY2_85.0', 'DY2_-85.0', 'DY2_-75.0'])
TLWP_zrg_obs = TLWP_zrg_obs.drop(columns=['DY1_85.0', 'DY1_-85.0', 'DY1_-75.0', 'DY2_85.0', 'DY2_-85.0', 'DY2_-75.0'])
OSR_zrg_obs = OSR_zrg_obs.drop(columns=['DY1_85.0', 'DY1_-85.0', 'DY1_-75.0', 'DY2_85.0', 'DY2_-85.0', 'DY2_-75.0'])
OLR_zrg_obs = OLR_zrg_obs.drop(columns=['DY1_85.0', 'DY1_-85.0', 'DY1_-75.0', 'DY2_85.0', 'DY2_-85.0', 'DY2_-75.0'])

#### Ordering matters here
zrg_obs = pd.concat([PCP_zrg_obs, TLWP_zrg_obs, OSR_zrg_obs, OLR_zrg_obs], axis=1) 

In [ ]:
zrg_obs #should just be one row of the observations by 176 (no longer 200)

### Training preparation

In [ ]:
X_train = ppe_params

In [ ]:
train_run_labels = zrg_ppedataset.index.to_list()

In [ ]:
PCP_train = PCP_zrg_ppedataset #.loc[train_run_labels] #**(1/8)
TLWP_train = TLWP_zrg_ppedataset #.loc[train_run_labels] #**(1/4)
OSR_train = OSR_zrg_ppedataset #.loc[train_run_labels] #**(1/4)
OLR_train = OLR_zrg_ppedataset #.loc[train_run_labels] #**(1/8)

PCP_train.columns = PCP_zrg_ppedataset.columns.astype(str)
TLWP_train.columns = TLWP_zrg_ppedataset.columns.astype(str)
OSR_train.columns = OSR_zrg_ppedataset.columns.astype(str)
OLR_train.columns = OLR_zrg_ppedataset.columns.astype(str)

vars_train_list = [PCP_train, TLWP_train, OSR_train, OLR_train]

In [ ]:
#for random forest, no preprocessing will be used
Y_train_ZRG = np.stack((PCP_train, TLWP_train, OSR_train, OLR_train), axis = 0)
Y_train_ZRG = np.transpose(Y_train_ZRG, (1, 2, 0))
Y_train_ZRG.shape

In [ ]:
range_thl2tune = [0.1, 10]
range_qw2tune = [0.1, 10]
range_length_fac = [0.1, 10]
range_c_diag_3rd_mom = [0.1, 10]
range_Ckh = [0.1, 1]
range_Ckm = [0.1, 1]
range_lambda_low = [0.0001, 0.1]
range_lambda_high = [0.0001, 0.1]
range_spa_to_nc = [0.1, 10]
range_p3_eci = [0.1, 1]
range_p3_eri = [0.1, 1]
range_k_acc = [0.01, 100]
range_p3_dep_nucleation_exponent = [0.2, 0.304]
range_max_total_ni = [5e5, 1e7]
#range_ice_sed_knob = [0.1, 2] #restricted range more appropriately below
range_ice_sed_knob = [1, 2]
range_p3_d_breakup_cutoff = [0, 500e-6]

dict_range_pars = dict()
dict_range_pars['length_fac'] = range_length_fac
dict_range_pars['p3_spa_to_nc'] = range_spa_to_nc
dict_range_pars['p3_k_accretion'] = range_k_acc
dict_range_pars['p3_ice_sed_knob'] = range_ice_sed_knob
dict_range_pars['thl2tune'] = range_thl2tune
dict_range_pars['qw2tune'] = range_qw2tune
dict_range_pars['c_diag_3rd_mom'] = range_c_diag_3rd_mom
dict_range_pars['Ckh'] = range_Ckh
dict_range_pars['Ckm'] = range_Ckm
dict_range_pars['lambda_low'] = range_lambda_low
dict_range_pars['lambda_high'] = range_lambda_high
dict_range_pars['p3_eci'] = range_p3_eci
dict_range_pars['p3_eri'] = range_p3_eri
dict_range_pars['p3_dep_nucleation_exponent'] = range_p3_dep_nucleation_exponent
dict_range_pars['p3_d_breakup_cutoff'] = range_p3_d_breakup_cutoff
dict_range_pars['max_total_ni'] = range_max_total_ni

In [ ]:
#Create bounds for minmax scaler -- using full bounds of parameters
param_bounds = np.array([dict_range_pars[param] for param in ppe_params.columns])

In [ ]:
#transform data
X_pipe_sk_minmax = preprocessing.MinMaxScaler()
X_pipe_sk_minmax.fit(param_bounds.T)  # fit on lower and upper bounds (2,16)
X_train_norm = X_pipe_sk_minmax.transform(X_train)

#from scikitlearn
Y_pipe_sk_ss_PCP = preprocessing.StandardScaler()
Y_pipe_sk_ss_PCP.fit(PCP_train)
PCP_train_norm = Y_pipe_sk_ss_PCP.transform(PCP_train)

Y_pipe_sk_ss_TLWP = preprocessing.StandardScaler() #lots of other options for this: RobustScaler(), etc.
Y_pipe_sk_ss_TLWP.fit(TLWP_train)
TLWP_train_norm = Y_pipe_sk_ss_TLWP.transform(TLWP_train)

Y_pipe_sk_ss_OSR = preprocessing.StandardScaler()
Y_pipe_sk_ss_OSR.fit(OSR_train)
OSR_train_norm = Y_pipe_sk_ss_OSR.transform(OSR_train)

Y_pipe_sk_ss_OLR = preprocessing.StandardScaler()
Y_pipe_sk_ss_OLR.fit(OLR_train)
OLR_train_norm = Y_pipe_sk_ss_OLR.transform(OLR_train)

Y_train_norm = np.stack((PCP_train_norm, TLWP_train_norm, OSR_train_norm, OLR_train_norm), axis = 0)
Y_train_norm = np.transpose(Y_train_norm, (1, 2, 0))
print(X_train_norm.shape, Y_train_norm.shape)

#### Employing models

In [ ]:
print(X_train_norm.shape, Y_train_norm.shape)

##### Model

In [ ]:
model_gp = gp_model(X_train_norm, Y_train_norm)
#model_cnn = cnn_model(X_train_norm, Y_train_norm)
#model_rf = rf_model(X_train.to_numpy(), Y_train_ZRG) #no preprocessing needed

In [ ]:
model_gp.train() #train the model

#### Saving transforms

In [ ]:
### Saving projections both normed and not and R2s
timestamp_day = datetime.now().strftime("%Y-%m-%d")
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
#path - /global/cfs/cdirs/e3sm/jpaige3/ESEm/TF_saving/ --- REPLACE THIS PATH TO SAVE THE PIPELINE WITH THE MODEL
GP_proj_filename = f"{paths.ZRG_DIR}/GP_ZRG_masked_proj_{timestamp}.pkl"

#make dictionary to save
GP_data_to_save = {
    'X_pipeline': X_pipe_sk_minmax, 
    'Y_pipeline_PCP': Y_pipe_sk_ss_PCP, 
    'Y_pipeline_TLWP': Y_pipe_sk_ss_TLWP, 
    'Y_pipeline_OSR': Y_pipe_sk_ss_OSR, 
    'Y_pipeline_OLR': Y_pipe_sk_ss_OLR,
    ####
    'X_train_index': train_run_labels,
    ### normalized/transformed
    'X_train_norm': X_train_norm, 
    'Y_train_norm': Y_train_norm,
    'PCP_train_norm': PCP_train_norm, 
    'TLWP_train_norm': TLWP_train_norm, 
    'OSR_train_norm': OSR_train_norm, 
    'OLR_train_norm':OLR_train_norm,
    ### unnormalized/untransformed
    'X_train': X_train, #ppe_params #.to_numpy()
    'Y_train': Y_train_ZRG,
    'PCP_train': PCP_train, 
    'TLWP_train': TLWP_train, 
    'OSR_train': OSR_train, 
    'OLR_train': OLR_train,
}

#save using pickle
with open(GP_proj_filename, 'wb') as f:
    pickle.dump(GP_data_to_save, f)

In [ ]:
### Saving obs
#path - /global/cfs/cdirs/e3sm/jpaige3/ESEm/GP_Saved_Model_Data/ --- REPLACE THIS PATH TO SAVE THE PIPELINE WITH THE MODEL
obs_save_filename = f"{paths.ZRG_DIR}/obs_{timestamp}.pkl"

#make dictionary to save
obs_data_to_save = {
    'zrg_obs': zrg_obs, 
    'PCP_zrg_obs': PCP_zrg_obs, 
    'TLWP_zrg_obs': TLWP_zrg_obs, 
    'OSR_zrg_obs': OSR_zrg_obs, 
    'OLR_zrg_obs': OLR_zrg_obs,
}

#save using pickle
with open(obs_save_filename, 'wb') as f:
    pickle.dump(obs_data_to_save, f)